# VAD Result

> Standardized result DTO for the voice-activity-detection task — the data noun VAD tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

**Canonical home (execution stage 8 — Option C / PILLAR 1c):** `VADResult` lives here in `cjm-capability-primitives` so a pure-compute VAD tool capability depends only on this data noun, never on the adapter machinery (`TaskAdapter`, the tool protocol, persistence). It is the task-keyed successor to the fused-era generic `MediaAnalysisResult` (silero-vad was the sole `MediaAnalysisPlugin`); the wire kind is `"vad.result"`. Because `ranges` holds typed `TimeRange` objects, a custom `from_dict` re-types them on wire-decode (the auto flat reconstruct would leave them as dicts).

In [ ]:
#| default_exp vad

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional

from cjm_plugin_system.core.wire import wire_type

In [ ]:
#| export
@dataclass
class TimeRange:
    """A temporal segment within an audio source (the VAD speech/silence span)."""
    start: float                                           # Start time in seconds
    end: float                                             # End time in seconds
    label: str = "speech"                                  # Segment type (e.g. 'speech')
    confidence: Optional[float] = None                     # Detection confidence (0.0 to 1.0)
    payload: Dict[str, Any] = field(default_factory=dict)  # Extra data (reserved)

    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

In [ ]:
#| export
@wire_type("vad.result")
@dataclass
class VADResult:
    """Standardized output for voice-activity-detection capabilities."""
    ranges: List[TimeRange]                                 # Detected speech segments, sorted by start
    metadata: Dict[str, Any] = field(default_factory=dict)  # Global VAD stats (duration, sample_rate, total_speech, ...)

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any],  # Wire payload (the substrate envelope's "data")
    ) -> "VADResult":  # Reconstructed result with typed ranges
        """Reconstruct from a wire payload, re-typing nested TimeRanges.

        `ranges` holds typed `TimeRange` objects, so the substrate's typed wire
        envelope (stage 2) reconstructs them host-side here rather than leaving
        bare dicts (which would break attribute access like `r.start`).
        """
        return cls(
            ranges=[r if isinstance(r, TimeRange) else TimeRange(**r)
                    for r in (d.get("ranges") or [])],
            metadata=d.get("metadata") or {},
        )

In [ ]:
# Test VADResult
result = VADResult(
    ranges=[
        TimeRange(start=0.0, end=2.5, label="speech", confidence=1.0),
        TimeRange(start=3.1, end=5.8, label="speech", confidence=1.0),
    ],
    metadata={"duration": 6.0, "sample_rate": 16000, "total_speech": 5.2, "segment_count": 2},
)

print(f"Ranges: {result.ranges}")
print(f"Metadata: {result.metadata}")
assert result.ranges[0].start == 0.0 and result.ranges[1].end == 5.8

In [ ]:
# Test minimal result (empty ranges = silent source)
minimal = VADResult(ranges=[])
print(f"Minimal result: {minimal}")
assert minimal.ranges == [] and minimal.metadata == {}

In [ ]:
# Wire-format executable test: the result round-trips TYPED through the
# simulated worker boundary (encode -> json -> decode), and the custom
# from_dict re-types the nested TimeRanges (NOT left as bare dicts).
import json as _json
from cjm_plugin_system.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

res = VADResult(
    ranges=[TimeRange(start=0.0, end=2.5), TimeRange(start=3.1, end=5.8)],
    metadata={"duration": 6.0, "sample_rate": 16000},
)
envelope = wire_encode(res)
assert envelope[WIRE_KIND_KEY] == "vad.result"
back = wire_decode(_json.loads(_json.dumps(envelope)))
assert isinstance(back, VADResult)
assert all(isinstance(r, TimeRange) for r in back.ranges)
assert back == res
print("vad.result wire round-trip OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()